# Premier League Goalkeeper Analysis 2025–26

Statistical analysis of all Premier League goalkeepers with **900+ minutes played**, through Gameday 31 of the 2025–26 season.

**8 analyses covered:**
1. Save Percentage
2. Goals Prevented (xG-based, diverging bar)
3. Save % Inside vs Outside Box
4. Clean Sheet Percentage
5. Error Rate per 90 mins
6. Aerial Dominance (High Claims per 90)
7. Composite Ranking (weighted score)
8. Scatter: Save % vs Goals Prevented

**Composite ranking weights:** Save% (30%) · Goals prevented/90 (30%) · Clean sheet% (20%) · Error rate (10%, inverted) · Rating (10%)

## Setup — Imports & Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os

os.makedirs("charts", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)

CSV_PATH = "data/premier_league_stats_gw31.csv"
MIN_MINUTES = 900

df = pd.read_csv(CSV_PATH, encoding="latin-1")
gks = df[df["position"] == "G"].copy()
gks = gks[gks["minutesPlayed"].astype(float) >= MIN_MINUTES].copy()

NUMERIC = [
    "saves", "goalsConceded", "goalsPrevented", "cleanSheet", "matchesStarted",
    "minutesPlayed", "penaltySave", "penaltyFaced", "errorLeadToGoal",
    "errorLeadToShot", "highClaims", "punches", "rating", "appearances",
    "savedShotsFromInsideTheBox", "savedShotsFromOutsideTheBox",
    "goalsConcededInsideTheBox", "goalsConcededOutsideTheBox",
]
for col in NUMERIC:
    gks[col] = pd.to_numeric(gks[col], errors="coerce")

gks["per90"] = gks["minutesPlayed"] / 90
gks["save_pct"] = gks["saves"] / (gks["saves"] + gks["goalsConceded"]) * 100
gks["clean_sheet_pct"] = gks["cleanSheet"] / gks["matchesStarted"] * 100
gks["inside_save_pct"] = (
    gks["savedShotsFromInsideTheBox"]
    / (gks["savedShotsFromInsideTheBox"] + gks["goalsConcededInsideTheBox"])
    * 100
)
gks["outside_save_pct"] = (
    gks["savedShotsFromOutsideTheBox"]
    / (gks["savedShotsFromOutsideTheBox"] + gks["goalsConcededOutsideTheBox"])
    * 100
)
gks["errors_per90"] = (gks["errorLeadToGoal"] + gks["errorLeadToShot"]) / gks["per90"]
gks["high_claims_per90"] = gks["highClaims"] / gks["per90"]
gks["prevented_per90"] = gks["goalsPrevented"] / gks["per90"]
gks["label"] = (
    gks["player_name"] + "\n(" + gks["team_name"].str.replace("&amp;", "&") + ")"
)

print(f"Qualifying GKs: {len(gks)}")
gks[["player_name", "team_name", "minutesPlayed", "save_pct", "goalsPrevented", "cleanSheet"]].sort_values("save_pct", ascending=False)

## Chart 1: Save Percentage

Overall save rate for each qualifying GK. **Green** = above 70%, **red** = below 70%. The dashed navy line shows the league average.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
data = gks.sort_values("save_pct", ascending=True)
colors = ["#2ecc71" if v >= 70 else "#e74c3c" for v in data["save_pct"]]
bars = ax.barh(data["label"], data["save_pct"], color=colors)
ax.axvline(data["save_pct"].mean(), color="navy", linestyle="--", linewidth=1.2,
           label=f"Avg: {data['save_pct'].mean():.1f}%")
ax.bar_label(bars, fmt="%.1f%%", padding=4, fontsize=8)
ax.set_xlabel("Save Percentage (%)")
ax.set_title("Save Percentage — GKs with 900+ mins (PL 2025–26, GW31)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("charts/01_save_percentage.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 2: Goals Prevented (xG-based)

Positive = GK saved more goals than statistically expected. Negative = conceded more than expected. Based on Post-Shot xG model.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
data = gks.dropna(subset=["goalsPrevented"]).sort_values("goalsPrevented", ascending=True)
colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in data["goalsPrevented"]]
bars = ax.barh(data["label"], data["goalsPrevented"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.bar_label(bars, fmt="%.2f", padding=4, fontsize=8)
ax.set_xlabel("Goals Prevented (positive = better than expected)")
ax.set_title("Goals Prevented Above/Below xG — PL 2025–26, GW31", fontweight="bold")
plt.tight_layout()
plt.savefig("charts/02_goals_prevented.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 3: Save % Inside vs Outside Box

Inside-box saves are harder — GKs face closer, better-quality shots. Outside-box save % tends to be higher because those shots are more speculative.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
data = gks.dropna(subset=["inside_save_pct", "outside_save_pct"]).sort_values(
    "inside_save_pct", ascending=True
)
y = np.arange(len(data))
width = 0.38
ax.barh(y - width / 2, data["inside_save_pct"], width, label="Inside Box", color="#3498db")
ax.barh(y + width / 2, data["outside_save_pct"], width, label="Outside Box", color="#e67e22")
ax.set_yticks(y)
ax.set_yticklabels(data["label"], fontsize=8)
ax.set_xlabel("Save Percentage (%)")
ax.set_title("Save % — Inside Box vs Outside Box | PL 2025–26, GW31", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("charts/03_inside_outside_save_pct.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 4: Clean Sheet Percentage

Clean sheet rate per GK. **Green** >= 30%, **orange** >= 20%, **red** < 20%. Partially reflects team defensive quality, not just GK performance.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
data = gks.sort_values("clean_sheet_pct", ascending=True)
colors = ["#2ecc71" if v >= 30 else "#e67e22" if v >= 20 else "#e74c3c"
          for v in data["clean_sheet_pct"]]
bars = ax.barh(data["label"], data["clean_sheet_pct"], color=colors)
ax.axvline(data["clean_sheet_pct"].mean(), color="navy", linestyle="--", linewidth=1.2,
           label=f"Avg: {data['clean_sheet_pct'].mean():.1f}%")
ax.bar_label(bars, fmt="%.1f%%", padding=4, fontsize=8)
ax.set_xlabel("Clean Sheet %")
ax.set_title("Clean Sheet Percentage — GKs with 900+ mins | PL 2025–26, GW31",
             fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("charts/04_clean_sheet_pct.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 5: Error Rate per 90 Minutes

Errors leading to a goal or a shot on target, normalized per 90 minutes. Lower is better. **Red** > 0.10/90, **orange** > 0.05/90, **green** <= 0.05/90.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
data = gks.sort_values("errors_per90", ascending=False)
colors = ["#e74c3c" if v > 0.1 else "#e67e22" if v > 0.05 else "#2ecc71"
          for v in data["errors_per90"]]
bars = ax.barh(data["label"], data["errors_per90"], color=colors)
ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=8)
ax.set_xlabel("Errors (leading to goal or shot) per 90 mins")
ax.set_title("GK Error Rate per 90 mins — Lower is Better | PL 2025–26, GW31",
             fontweight="bold")
plt.tight_layout()
plt.savefig("charts/05_error_rate_per90.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 6: Aerial Dominance — High Claims per 90

Number of aerial balls claimed per 90 minutes. A key indicator of a GK's command of the penalty area and dominance in the air.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
data = gks.sort_values("high_claims_per90", ascending=True)
bars = ax.barh(data["label"], data["high_claims_per90"], color="#9b59b6")
ax.axvline(data["high_claims_per90"].mean(), color="navy", linestyle="--", linewidth=1.2,
           label=f"Avg: {data['high_claims_per90'].mean():.2f}")
ax.bar_label(bars, fmt="%.2f", padding=4, fontsize=8)
ax.set_xlabel("High Claims per 90 mins")
ax.set_title("Aerial Dominance — High Claims per 90 | PL 2025–26, GW31",
             fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("charts/06_high_claims_per90.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 7: Composite Ranking

Weighted composite score combining 5 metrics:

| Metric | Weight | Direction |
|--------|--------|-----------|
| Save % | 30% | Higher = better |
| Goals prevented per 90 | 30% | Higher = better |
| Clean sheet % | 20% | Higher = better |
| Error rate per 90 | 10% | Lower = better (inverted) |
| Match rating | 10% | Higher = better |

Each metric is min-max normalized to 0–1 before weighting.

In [ ]:
rank_df = gks.dropna(subset=["save_pct", "prevented_per90", "clean_sheet_pct",
                              "errors_per90", "rating"]).copy()

def norm(series, invert=False):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series([0.5] * len(series), index=series.index)
    normalized = (series - mn) / (mx - mn)
    return 1 - normalized if invert else normalized

rank_df["score"] = (
    norm(rank_df["save_pct"])          * 0.30
    + norm(rank_df["prevented_per90"]) * 0.30
    + norm(rank_df["clean_sheet_pct"]) * 0.20
    + norm(rank_df["errors_per90"], invert=True) * 0.10
    + norm(rank_df["rating"])          * 0.10
)

rank_df = rank_df.sort_values("score", ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(rank_df["label"], rank_df["score"], color="#2980b9")
ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=8)
ax.set_xlabel("Composite Score (0–1)")
ax.set_title(
    "Composite GK Ranking (Save% 30% | xG-prevented 30% | CS% 20% | Errors 10% | Rating 10%)\nPL 2025–26, GW31",
    fontweight="bold",
)
plt.tight_layout()
plt.savefig("charts/07_composite_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

top = rank_df[["player_name", "team_name", "save_pct", "goalsPrevented",
               "clean_sheet_pct", "errors_per90", "rating", "score"]
              ].sort_values("score", ascending=False).copy()
top.columns = ["Player", "Team", "Save%", "Prevented", "CS%", "Err/90", "Rating", "Score"]
top["Score"] = top["Score"].round(3)
top

## Chart 8: Scatter — Save % vs Goals Prevented

Each bubble is a GK. **Bubble size** = appearances. **Colour** = match rating (green = high, red = low).

The top-right quadrant is the elite zone: high save % AND saving more goals than expected.

In [ ]:
scatter_df = gks.dropna(subset=["save_pct", "goalsPrevented"])
fig, ax = plt.subplots(figsize=(11, 8))
bubble_sizes = scatter_df["appearances"] * 12
sc = ax.scatter(
    scatter_df["save_pct"],
    scatter_df["goalsPrevented"],
    s=bubble_sizes,
    alpha=0.7,
    c=scatter_df["rating"],
    cmap="RdYlGn",
    edgecolors="black",
    linewidths=0.5,
)
for _, row in scatter_df.iterrows():
    ax.annotate(
        row["player_name"].split()[-1],
        (row["save_pct"], row["goalsPrevented"]),
        textcoords="offset points",
        xytext=(6, 4),
        fontsize=7.5,
    )
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.axvline(scatter_df["save_pct"].mean(), color="gray", linestyle="--", linewidth=0.8)
plt.colorbar(sc, ax=ax, label="Match Rating")
ax.set_xlabel("Save Percentage (%)")
ax.set_ylabel("Goals Prevented (xG-based)")
ax.set_title(
    "Save % vs Goals Prevented — bubble size = appearances, colour = rating\nTop-right quadrant = elite performance | PL 2025–26, GW31",
    fontweight="bold",
)
plt.tight_layout()
plt.savefig("charts/08_scatter_save_vs_prevented.png", dpi=150, bbox_inches="tight")
plt.show()